# Additional Local-Search Operators

This notebook evaluates whether additional local-search operators improve the solution after four applications of `move_plateau`.

In [27]:
from pathlib import Path

import numpy as np
import pandas as pd

## Configuration

In [28]:
GRAPH_ORDER = ["powerlaw", "er"]

DATASET_ORDER = [
    "small sparse",
    "small dense",
    "large sparse",
    "large dense",
]

BASE_PIPELINE = ",".join(["move_plateau"] * 4)

ADDITIONAL_OPERATORS = [
    "merge_first",
    "merge_best",
    "bridge_split",
    "split_min_cut",
]

PIPELINE_ORDER = [
    BASE_PIPELINE,
    *[f"{BASE_PIPELINE},{operator}" for operator in ADDITIONAL_OPERATORS],
]

RESULTS_DIR = Path("../../results/experiment2/extra_operator")
STEP_RESULTS_FILE = RESULTS_DIR / "step_results.csv"

## Load data

In [29]:
steps = pd.read_csv(STEP_RESULTS_FILE)

steps["dataset_group"] = steps["size_class"].astype(str) + " " + steps["regime"].astype(str)

steps["graph_type"] = pd.Categorical(
    steps["graph_type"],
    categories=GRAPH_ORDER,
    ordered=True,
)

steps["dataset_group"] = pd.Categorical(
    steps["dataset_group"],
    categories=DATASET_ORDER,
    ordered=True,
)

steps = steps[steps["pipeline"].isin(PIPELINE_ORDER) & (steps["start_partition"] == "maximum_matching")].copy()

steps["pipeline"] = pd.Categorical(
    steps["pipeline"],
    categories=PIPELINE_ORDER,
    ordered=True,
)

In [30]:
execution_keys = [
    "graph_type",
    "dataset_group",
    "dataset",
    "instance",
    "pipeline",
    "run",
]

final_steps = (
    steps
    .sort_values(execution_keys + ["step_index"])
    .groupby(execution_keys, observed=True, as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

best_run_keys = [
    "graph_type",
    "dataset_group",
    "dataset",
    "instance",
    "pipeline",
]

best_runs = (
    final_steps
    .sort_values(["score_after", "runtime", "run"], ascending=[False, True, True])
    .groupby(best_run_keys, observed=True, as_index=False)
    .head(1)
    .reset_index(drop=True)
)

## Relative solution quality

For every instance and pipeline, only the run with the highest final solution quality is retained.

The pipeline consisting only of four applications of `move_plateau` serves as the reference. The relative solution quality of an extended pipeline is therefore defined as

$
\frac{\text{solution quality after four applications of } \texttt{move\_plateau}}
     {\text{solution quality of the extended pipeline}}.
$

A value of $1.0$ indicates that the additional operator does not change the final solution quality.
Values smaller than $1.0$ indicate that the additional operator improves the final solution.

In [31]:
instance_keys = [
    "graph_type",
    "dataset_group",
    "dataset",
    "instance",
]

density_table = best_runs.pivot(
    index=instance_keys,
    columns="pipeline",
    values="score_after",
)[PIPELINE_ORDER]

reference_score = density_table[BASE_PIPELINE]
relative_to_reference = density_table.rdiv(reference_score, axis=0)

relative_quality_summary = (
    relative_to_reference
    .groupby(level=["graph_type", "dataset_group"])
    .mean()
    .stack()
    .rename("mean_relative_quality_to_reference")
    .reset_index()
)

relative_quality_summary["operator"] = (
    relative_quality_summary["pipeline"]
    .astype(str)
    .str.rsplit(",", n=1)
    .str[-1]
)

relative_quality_summary = (
    relative_quality_summary[relative_quality_summary["operator"].isin(ADDITIONAL_OPERATORS)].drop(columns="pipeline")
)

relative_quality_summary["operator"] = pd.Categorical(
    relative_quality_summary["operator"],
    categories=ADDITIONAL_OPERATORS,
    ordered=True,
)

relative_quality_summary = (
    relative_quality_summary
    .sort_values(["graph_type", "dataset_group", "operator"])
    .reset_index(drop=True)
)

relative_quality_summary

,graph_type,dataset_group,mean_relative_quality_to_reference,operator
0,powerlaw,small sparse,0.999963,merge_first
1,powerlaw,small sparse,0.999963,merge_best
2,powerlaw,small sparse,0.999958,bridge_split
3,powerlaw,small sparse,1.000000,split_min_cut
4,powerlaw,small dense,0.999973,merge_first
5,powerlaw,small dense,0.999973,merge_best
6,powerlaw,small dense,1.000000,bridge_split
7,powerlaw,small dense,1.000000,split_min_cut
8,powerlaw,large sparse,1.000000,merge_first
9,powerlaw,large sparse,1.000000,merge_best


## Operator activity and run time

Operator activity is measured on the selected best runs. Run time is summed over all randomized runs for each instance and then averaged over the corresponding dataset group.

In [32]:
best_additional_steps = best_runs[best_runs["step_name"].isin(ADDITIONAL_OPERATORS)]

activity_summary = (
    best_additional_steps
    .groupby(["graph_type", "dataset_group", "step_name"], observed=True, as_index=False)
    .agg(
        affected_instances=("num_moves", lambda s: (s > 0).sum()),
        affected_instance_percent=("num_moves", lambda s: 100 * (s > 0).mean()),
        total_moves=("num_moves", "sum"),
    )
    .rename(columns={"step_name": "operator"})
)

In [33]:
all_additional_steps = final_steps[final_steps["step_name"].isin(ADDITIONAL_OPERATORS)]

runtime_per_instance = (
    all_additional_steps
    .groupby(["graph_type", "dataset_group", "dataset", "instance", "step_name"], observed=True, as_index=False)
    .agg(total_operator_runtime=("runtime", "sum"))
)

runtime_summary = (
    runtime_per_instance
    .groupby(["graph_type", "dataset_group", "step_name"], observed=True, as_index=False)
    .agg(mean_total_operator_runtime=("total_operator_runtime", "mean"))
    .rename(columns={"step_name": "operator"})
)

In [34]:
operator_summary = activity_summary.merge(runtime_summary, on=["graph_type", "dataset_group", "operator"])

operator_summary["operator"] = pd.Categorical(operator_summary["operator"], categories=ADDITIONAL_OPERATORS, ordered=True)

operator_summary = (
    operator_summary
    .sort_values(["graph_type", "dataset_group", "operator"])
    .reset_index(drop=True)
)

operator_summary

,graph_type,dataset_group,operator,affected_instances,affected_instance_percent,total_moves,mean_total_operator_runtime
0,powerlaw,small sparse,merge_first,2,0.8,2,0.101804
1,powerlaw,small sparse,merge_best,2,0.8,2,0.098959
2,powerlaw,small sparse,bridge_split,3,1.2,3,0.109508
3,powerlaw,small sparse,split_min_cut,0,0.0,0,0.116094
4,powerlaw,small dense,merge_first,4,1.6,4,0.135439
5,powerlaw,small dense,merge_best,4,1.6,4,0.131186
6,powerlaw,small dense,bridge_split,0,0.0,0,0.130072
7,powerlaw,small dense,split_min_cut,0,0.0,0,0.142094
8,powerlaw,large sparse,merge_first,0,0.0,0,0.826063
9,powerlaw,large sparse,merge_best,0,0.0,0,0.848652


## LaTeX helper functions

In [35]:
def truncate_number(value: float, decimals: int) -> float:
    factor = 10 ** decimals
    return np.trunc(value * factor) / factor

def latex_operator(operator: str) -> str:
    return r"\texttt{" + operator.replace("_", r"\_") + "}"

def format_number(value: float, decimals: int) -> str:
    return f"{value:.{decimals}f}"

def format_percent(value: float, decimals: int = 1) -> str:
    return rf"{truncate_number(value, decimals):.{decimals}f}\,\%"

## Build LaTeX tables

In [42]:
def make_quality_latex_table(df: pd.DataFrame, graph_type: str, caption: str, label: str) -> str:
    graph_df = df[df["graph_type"] == graph_type]

    lines = [
        r"\begin{table}[!htbp]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{lp{4.0cm}r}",
        r"\toprule",
        r"Dataset & Operator & \shortstack{Mean quality\\ratio} \\",
        r"\midrule",
    ]

    for dataset_index, dataset in enumerate(DATASET_ORDER):
        part = graph_df[graph_df["dataset_group"] == dataset].sort_values("operator")

        best_quality = part["mean_relative_quality_to_reference"].min()

        for row_index, row in enumerate(part.itertuples(index=False)):
            dataset_cell = (
                rf"\multirow{{{len(part)}}}{{*}}{{{dataset}}}"
                if row_index == 0
                else ""
            )

            quality = format_number(row.mean_relative_quality_to_reference, 6)

            if (row.mean_relative_quality_to_reference < 1.0 and np.isclose(row.mean_relative_quality_to_reference, best_quality)):
                quality = rf"\textbf{{{quality}}}"

            lines.append(
                f"{dataset_cell} "
                f"& {latex_operator(str(row.operator))} "
                f"& {quality} "
                r"\\"
            )

        if dataset_index < len(DATASET_ORDER) - 1:
            lines.append(r"\cmidrule(l){1-3}")

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

In [43]:
powerlaw_quality_latex = make_quality_latex_table(
    relative_quality_summary,
    graph_type="powerlaw",
    caption=(
        "Mean quality ratio of the additional local-search operators on Powerlaw instances relative to the reference pipeline consisting of four applications of \\texttt{move\_plateau}. Values below 1 indicate an improvement over the reference."
    ),
    label="tab:additional_operator_quality_powerlaw",
)

print(powerlaw_quality_latex)

\begin{table}[!htbp]
\centering
\caption{Mean quality ratio of the additional local-search operators on Powerlaw instances relative to the reference pipeline consisting of four applications of \texttt{move\_plateau}. Values below 1 indicate an improvement over the reference.}
\label{tab:additional_operator_quality_powerlaw}
\begin{tabular}{lp{4.0cm}r}
\toprule
Dataset & Operator & \shortstack{Mean quality\\ratio} \\
\midrule
\multirow{4}{*}{small sparse} & \texttt{merge\_first} & \textbf{0.999963} \\
 & \texttt{merge\_best} & \textbf{0.999963} \\
 & \texttt{bridge\_split} & \textbf{0.999958} \\
 & \texttt{split\_min\_cut} & 1.000000 \\
\cmidrule(l){1-3}
\multirow{4}{*}{small dense} & \texttt{merge\_first} & \textbf{0.999973} \\
 & \texttt{merge\_best} & \textbf{0.999973} \\
 & \texttt{bridge\_split} & 1.000000 \\
 & \texttt{split\_min\_cut} & 1.000000 \\
\cmidrule(l){1-3}
\multirow{4}{*}{large sparse} & \texttt{merge\_first} & 1.000000 \\
 & \texttt{merge\_best} & 1.000000 \\
 & \textt

In [44]:
er_quality_latex = make_quality_latex_table(
    relative_quality_summary,
    graph_type="er",
    caption=(
        "Mean quality ratio of the additional local-search operators on Erdős-Rényi instances relative to the reference pipeline consisting of four applications of \\texttt{move\\_plateau}. Values below 1 indicate an improvement over the reference."
    ),
    label="tab:additional_operator_quality_er",
)

print(er_quality_latex)

\begin{table}[!htbp]
\centering
\caption{Mean quality ratio of the additional local-search operators on Erdős-Rényi instances relative to the reference pipeline consisting of four applications of \texttt{move\_plateau}. Values below 1 indicate an improvement over the reference.}
\label{tab:additional_operator_quality_er}
\begin{tabular}{lp{4.0cm}r}
\toprule
Dataset & Operator & \shortstack{Mean quality\\ratio} \\
\midrule
\multirow{4}{*}{small sparse} & \texttt{merge\_first} & 1.000000 \\
 & \texttt{merge\_best} & 1.000000 \\
 & \texttt{bridge\_split} & \textbf{0.999953} \\
 & \texttt{split\_min\_cut} & 1.000000 \\
\cmidrule(l){1-3}
\multirow{4}{*}{small dense} & \texttt{merge\_first} & \textbf{0.999977} \\
 & \texttt{merge\_best} & \textbf{0.999977} \\
 & \texttt{bridge\_split} & 1.000000 \\
 & \texttt{split\_min\_cut} & 1.000000 \\
\cmidrule(l){1-3}
\multirow{4}{*}{large sparse} & \texttt{merge\_first} & 1.000000 \\
 & \texttt{merge\_best} & 1.000000 \\
 & \texttt{bridge\_split} & 1.

In [45]:
def make_activity_latex_table(df: pd.DataFrame, graph_type: str, caption: str, label: str) -> str:
    graph_df = df[df["graph_type"] == graph_type]

    lines = [
        r"\begin{table}[!htbp]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{lp{3.4cm}rrrr}",
        r"\toprule",
        r"Dataset & Operator & \shortstack{Affected\\instances} & Rate & Moves & \shortstack{Additional\\run time (s)} \\",
        r"\midrule",
    ]

    for dataset_index, dataset in enumerate(DATASET_ORDER):
        part = graph_df[graph_df["dataset_group"] == dataset].sort_values("operator")

        for row_index, row in enumerate(part.itertuples(index=False)):
            dataset_cell = (
                rf"\multirow{{{len(part)}}}{{*}}{{{dataset}}}"
                if row_index == 0
                else ""
            )

            lines.append(
                f"{dataset_cell} "
                f"& {latex_operator(str(row.operator))} "
                f"& {int(row.affected_instances)} "
                f"& {format_percent(row.affected_instance_percent, 1)} "
                f"& {int(row.total_moves)} "
                f"& {format_number(row.mean_total_operator_runtime, 4)} "
                r"\\"
            )

        if dataset_index < len(DATASET_ORDER) - 1:
            lines.append(r"\cmidrule(l){1-6}")

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

In [48]:
powerlaw_activity_latex = make_activity_latex_table(
    operator_summary,
    graph_type="powerlaw",
    caption=(
        "Activity and mean run time of the additional operators on Powerlaw instances."
    ),
    label="tab:additional_operator_activity_powerlaw",
)

print(powerlaw_activity_latex)

\begin{table}[!htbp]
\centering
\caption{Activity and mean run time of the additional operators on Powerlaw instances.}
\label{tab:additional_operator_activity_powerlaw}
\begin{tabular}{lp{3.4cm}rrrr}
\toprule
Dataset & Operator & \shortstack{Affected\\instances} & Rate & Moves & \shortstack{Additional\\run time (s)} \\
\midrule
\multirow{4}{*}{small sparse} & \texttt{merge\_first} & 2 & 0.8\,\% & 2 & 0.1018 \\
 & \texttt{merge\_best} & 2 & 0.8\,\% & 2 & 0.0990 \\
 & \texttt{bridge\_split} & 3 & 1.2\,\% & 3 & 0.1095 \\
 & \texttt{split\_min\_cut} & 0 & 0.0\,\% & 0 & 0.1161 \\
\cmidrule(l){1-6}
\multirow{4}{*}{small dense} & \texttt{merge\_first} & 4 & 1.6\,\% & 4 & 0.1354 \\
 & \texttt{merge\_best} & 4 & 1.6\,\% & 4 & 0.1312 \\
 & \texttt{bridge\_split} & 0 & 0.0\,\% & 0 & 0.1301 \\
 & \texttt{split\_min\_cut} & 0 & 0.0\,\% & 0 & 0.1421 \\
\cmidrule(l){1-6}
\multirow{4}{*}{large sparse} & \texttt{merge\_first} & 0 & 0.0\,\% & 0 & 0.8261 \\
 & \texttt{merge\_best} & 0 & 0.0\,\% & 0 & 0.

In [49]:
er_activity_latex = make_activity_latex_table(
    operator_summary,
    graph_type="er",
    caption=(
        "Activity and mean run time of the additional operators on Erdős-Rényi instances."
    ),
    label="tab:additional_operator_activity_er",
)

print(er_activity_latex)

\begin{table}[!htbp]
\centering
\caption{Activity and mean run time of the additional operators on Erdős-Rényi instances.}
\label{tab:additional_operator_activity_er}
\begin{tabular}{lp{3.4cm}rrrr}
\toprule
Dataset & Operator & \shortstack{Affected\\instances} & Rate & Moves & \shortstack{Additional\\run time (s)} \\
\midrule
\multirow{4}{*}{small sparse} & \texttt{merge\_first} & 0 & 0.0\,\% & 0 & 0.1085 \\
 & \texttt{merge\_best} & 1 & 0.4\,\% & 1 & 0.1039 \\
 & \texttt{bridge\_split} & 1 & 0.4\,\% & 1 & 0.0983 \\
 & \texttt{split\_min\_cut} & 0 & 0.0\,\% & 0 & 0.1019 \\
\cmidrule(l){1-6}
\multirow{4}{*}{small dense} & \texttt{merge\_first} & 4 & 1.6\,\% & 4 & 0.1515 \\
 & \texttt{merge\_best} & 4 & 1.6\,\% & 4 & 0.1447 \\
 & \texttt{bridge\_split} & 0 & 0.0\,\% & 0 & 0.1210 \\
 & \texttt{split\_min\_cut} & 0 & 0.0\,\% & 0 & 0.1294 \\
\cmidrule(l){1-6}
\multirow{4}{*}{large sparse} & \texttt{merge\_first} & 0 & 0.0\,\% & 0 & 0.8984 \\
 & \texttt{merge\_best} & 0 & 0.0\,\% & 0 & 0.921